# Guardrail Model Evaluation

Evaluate saved guardrail classifier models under `model/` against:

- Synthetic dataset: `dataset/fahmai_guardrail_bert_all.csv`
- Real dataset: `dataset/test/questions_formatted_id.csv`

The notebook normalizes both supported dataset schemas:

- New schema: `text`, `label`, `category`, `source_file`, `source_id`
- Legacy schema: `Id`, `Instruct`, `Label`, `Category`

Results are saved under `outputs/evaluation/<timestamp>/`.


In [2]:
from __future__ import annotations

import json
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

sns.set_theme(style="whitegrid", context="notebook")


In [3]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent


@dataclass
class EvalConfig:
    model_root: str = "model"
    synthetic_path: str = "dataset/fahmai_guardrail_bert_all.csv"
    real_path: str = "dataset/test/questions_formatted_id.csv"
    task: str = "label"
    max_length: int = 256
    batch_size: int = 32
    output_root: str = "outputs/evaluation"
    device: str = "auto"


cfg = EvalConfig()

if cfg.task not in {"label", "category"}:
    raise ValueError("cfg.task must be either 'label' or 'category'")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if cfg.device != "auto":
    device = torch.device(cfg.device)

run_id = datetime.now().strftime("%Y%m%d-%H%M%S")
output_dir = PROJECT_ROOT / cfg.output_root / run_id
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {device}")
print(f"Task: {cfg.task}")
print(f"Output dir: {output_dir}")


Project root: c:\Users\Gunte\Workspace\Workspace\sides\spai-rag-hack-4\guardrail-pipeline
Device: cpu
Task: label
Output dir: c:\Users\Gunte\Workspace\Workspace\sides\spai-rag-hack-4\guardrail-pipeline\outputs\evaluation\20260603-014612


In [4]:
def resolve_path(path: str | Path) -> Path:
    path = Path(path)
    if path.is_absolute():
        return path
    return PROJECT_ROOT / path


def normalize_dataset(path: str | Path, dataset_name: str) -> pd.DataFrame:
    path = resolve_path(path)
    if not path.exists():
        raise FileNotFoundError(f"{dataset_name} dataset not found: {path}")

    frame = pd.read_csv(path, encoding="utf-8-sig")
    frame = frame.rename(
        columns={
            "Id": "source_id",
            "Instruct": "text",
            "Label": "label",
            "Category": "category",
        }
    )

    required_columns = {"text", cfg.task}
    missing_columns = required_columns.difference(frame.columns)
    if missing_columns:
        raise ValueError(f"{dataset_name} is missing required columns: {sorted(missing_columns)}")

    frame = frame.copy()
    frame["text"] = frame["text"].astype(str).str.strip()
    frame = frame[frame["text"].ne("")].reset_index(drop=True)

    if "label" in frame.columns:
        frame["label"] = pd.to_numeric(frame["label"], errors="raise").astype(int)
    if "category" in frame.columns:
        frame["category"] = frame["category"].astype(str).str.strip()
    if "source_file" not in frame.columns:
        frame["source_file"] = path.name
    if "source_id" not in frame.columns:
        frame["source_id"] = [f"{dataset_name}-{idx:06d}" for idx in range(len(frame))]

    frame["dataset"] = dataset_name
    frame["dataset_path"] = str(path)
    return frame


synthetic_df = normalize_dataset(cfg.synthetic_path, "synthetic")
real_df = normalize_dataset(cfg.real_path, "real")

print("Synthetic rows:", len(synthetic_df))
print(synthetic_df[cfg.task].value_counts(dropna=False).sort_index())
print("\nReal rows:", len(real_df))
print(real_df[cfg.task].value_counts(dropna=False).sort_index())


Synthetic rows: 7500
label
0    2335
1    5165
Name: count, dtype: int64

Real rows: 100
label
0    92
1     8
Name: count, dtype: int64


In [5]:
def discover_model_dirs(model_root: str | Path) -> list[Path]:
    model_root = resolve_path(model_root)
    if not model_root.exists():
        raise FileNotFoundError(f"Model root not found: {model_root}")

    model_dirs = []
    for config_path in model_root.rglob("config.json"):
        candidate = config_path.parent
        has_model_file = any(
            (candidate / filename).exists()
            for filename in ("model.safetensors", "pytorch_model.bin")
        )
        if has_model_file:
            model_dirs.append(candidate)

    return sorted(model_dirs)


model_dirs = discover_model_dirs(cfg.model_root)
if not model_dirs:
    raise FileNotFoundError(f"No saved Hugging Face model directories found under {cfg.model_root}")

for idx, model_dir in enumerate(model_dirs, start=1):
    print(f"{idx}. {model_dir.relative_to(PROJECT_ROOT)}")


1. model\wangchanberta\label\20260602-171753\model


In [6]:
def metric_dict(y_true: np.ndarray, y_pred: np.ndarray, id2label: dict[int, str]) -> dict:
    label_ids = sorted(id2label)
    label_names = [id2label[idx] for idx in label_ids]
    precision, recall, f1_weighted, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=label_ids,
        average="weighted",
        zero_division=0,
    )
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_weighted": float(precision),
        "recall_weighted": float(recall),
        "f1_weighted": float(f1_weighted),
        "f1_macro": float(f1_score(y_true, y_pred, labels=label_ids, average="macro", zero_division=0)),
        "labels": label_names,
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=label_ids).tolist(),
        "classification_report": classification_report(
            y_true,
            y_pred,
            labels=label_ids,
            target_names=label_names,
            zero_division=0,
            output_dict=True,
        ),
    }


def predict_frame(
    frame: pd.DataFrame,
    model,
    tokenizer,
    id2label: dict[int, str],
    label2id: dict[str, int],
    progress_label: str,
) -> tuple[pd.DataFrame, dict]:
    predictions = []
    probabilities = []

    texts = frame["text"].tolist()
    batch_starts = range(0, len(texts), cfg.batch_size)
    for start in tqdm(batch_starts, desc=progress_label, unit="batch"):
        batch_texts = texts[start : start + cfg.batch_size]
        encoded = tokenizer(
            batch_texts,
            truncation=True,
            max_length=cfg.max_length,
            padding=True,
            return_tensors="pt",
        )
        encoded = {key: value.to(device) for key, value in encoded.items()}

        with torch.no_grad():
            logits = model(**encoded).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()

        probabilities.append(probs)
        predictions.extend(np.argmax(probs, axis=-1).tolist())

    probs_all = np.vstack(probabilities)
    result = frame.copy()
    result["predicted_id"] = predictions
    result["predicted_label"] = [id2label[int(idx)] for idx in predictions]
    result["predicted_score"] = probs_all.max(axis=-1)

    for idx, label_name in id2label.items():
        result[f"score_{label_name}"] = probs_all[:, idx]

    true_labels = result[cfg.task].astype(str)
    known_mask = true_labels.isin(label2id)
    if not known_mask.all():
        dropped = sorted(true_labels[~known_mask].unique())
        print(f"Dropping rows with labels not in model label mapping: {dropped}")
        result = result[known_mask].reset_index(drop=True)
        true_labels = result[cfg.task].astype(str)

    y_true = true_labels.map(label2id).to_numpy()
    y_pred = result["predicted_id"].to_numpy()
    result["true_label"] = [id2label[int(idx)] for idx in y_true]
    result["is_wrong"] = result["true_label"] != result["predicted_label"]

    metrics = metric_dict(y_true, y_pred, id2label)
    metrics["rows"] = int(len(result))
    metrics["wrong_predictions"] = int(result["is_wrong"].sum())
    metrics["wrong_prediction_rate"] = float(result["is_wrong"].mean())
    return result, metrics


def load_label_maps(model) -> tuple[dict[int, str], dict[str, int]]:
    id2label = {int(key): str(value) for key, value in model.config.id2label.items()}
    label2id = {str(value): int(key) for key, value in id2label.items()}
    return id2label, label2id


In [ ]:
all_metrics = []

for model_dir in tqdm(model_dirs, desc="models", unit="model"):
    model_name = str(model_dir.relative_to(PROJECT_ROOT)).replace("\\", "/")
    model_output_dir = output_dir / model_name.replace("/", "__")
    model_output_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nEvaluating {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.to(device)
    model.eval()

    id2label, label2id = load_label_maps(model)

    datasets = [("synthetic", synthetic_df), ("real", real_df)]
    for dataset_name, frame in tqdm(datasets, desc=model_name, unit="dataset", leave=False):
        results, metrics = predict_frame(
            frame,
            model,
            tokenizer,
            id2label,
            label2id,
            progress_label=f"{dataset_name} batches",
        )
        wrong = results[results["is_wrong"]].copy()

        predictions_path = model_output_dir / f"{dataset_name}_predictions.csv"
        wrong_path = model_output_dir / f"{dataset_name}_wrong_predictions.csv"
        metrics_path = model_output_dir / f"{dataset_name}_metrics.json"

        results.to_csv(predictions_path, index=False, encoding="utf-8-sig")
        wrong.to_csv(wrong_path, index=False, encoding="utf-8-sig")
        with metrics_path.open("w", encoding="utf-8") as handle:
            json.dump(metrics, handle, ensure_ascii=False, indent=2)

        metrics_row = {
            "model": model_name,
            "dataset": dataset_name,
            "predictions_path": str(predictions_path),
            "wrong_predictions_path": str(wrong_path),
            "metrics_path": str(metrics_path),
            **{
                key: value
                for key, value in metrics.items()
                if key
                in {
                    "rows",
                    "accuracy",
                    "precision_weighted",
                    "recall_weighted",
                    "f1_weighted",
                    "f1_macro",
                    "wrong_predictions",
                    "wrong_prediction_rate",
                }
            },
        }
        all_metrics.append(metrics_row)
        print(
            f"{dataset_name}: accuracy={metrics['accuracy']:.4f}, "
            f"f1_macro={metrics['f1_macro']:.4f}, wrong={metrics['wrong_predictions']}"
        )

summary_df = pd.DataFrame(all_metrics)
summary_path = output_dir / "evaluation_summary.csv"
summary_json_path = output_dir / "evaluation_summary.json"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
with summary_json_path.open("w", encoding="utf-8") as handle:
    json.dump(all_metrics, handle, ensure_ascii=False, indent=2)

summary_df


models:   0%|          | 0/1 [00:00<?, ?model/s]


Evaluating model/wangchanberta/label/20260602-171753/model


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model/wangchanberta/label/20260602-171753/model:   0%|          | 0/2 [00:00<?, ?dataset/s]

synthetic batches:   0%|          | 0/235 [00:00<?, ?batch/s]

In [ ]:
for row in all_metrics:
    wrong_path = Path(row["wrong_predictions_path"])
    wrong = pd.read_csv(wrong_path, encoding="utf-8-sig")
    print(f"\n{row['model']} | {row['dataset']} | wrong rows: {len(wrong)}")
    if wrong.empty:
        continue

    display_columns = [
        column
        for column in [
            "source_id",
            "source_file",
            "true_label",
            "predicted_label",
            "predicted_score",
            "text",
            "category",
        ]
        if column in wrong.columns
    ]
    display(wrong[display_columns].sort_values("predicted_score", ascending=False).head(50))


In [ ]:
if summary_df.empty:
    raise RuntimeError("No evaluation results available. Run the evaluation cell first.")

metric_columns = ["accuracy", "f1_macro", "precision_weighted", "recall_weighted"]
summary_long = summary_df.melt(
    id_vars=["model", "dataset"],
    value_vars=metric_columns,
    var_name="metric",
    value_name="value",
)

plt.figure(figsize=(12, 5))
sns.barplot(data=summary_long, x="metric", y="value", hue="dataset")
plt.ylim(0, 1)
plt.title("Evaluation Metrics by Dataset")
plt.xlabel("")
plt.ylabel("Score")
plt.legend(title="Dataset")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
sns.barplot(data=summary_df, x="dataset", y="wrong_prediction_rate", hue="model")
plt.ylim(0, max(0.05, summary_df["wrong_prediction_rate"].max() * 1.15))
plt.title("Wrong Prediction Rate")
plt.xlabel("Dataset")
plt.ylabel("Wrong prediction rate")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
sns.barplot(data=summary_df, x="dataset", y="wrong_predictions", hue="model")
plt.title("Wrong Prediction Count")
plt.xlabel("Dataset")
plt.ylabel("Rows")
plt.tight_layout()
plt.show()

for row in all_metrics:
    metrics = json.loads(Path(row["metrics_path"]).read_text(encoding="utf-8"))
    labels = metrics["labels"]
    matrix = np.array(metrics["confusion_matrix"])

    plt.figure(figsize=(5, 4))
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
    plt.title(f"Confusion Matrix\n{row['model']} | {row['dataset']}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

for row in all_metrics:
    predictions = pd.read_csv(row["predictions_path"], encoding="utf-8-sig")

    plt.figure(figsize=(10, 4))
    sns.histplot(
        data=predictions,
        x="predicted_score",
        hue="is_wrong",
        bins=30,
        multiple="stack",
    )
    plt.title(f"Prediction Confidence\n{row['model']} | {row['dataset']}")
    plt.xlabel("Predicted score")
    plt.ylabel("Rows")
    plt.tight_layout()
    plt.show()

    if "category" in predictions.columns:
        category_wrong = (
            predictions.groupby("category", dropna=False)["is_wrong"]
            .agg(rows="count", wrong_predictions="sum", wrong_prediction_rate="mean")
            .reset_index()
            .sort_values("wrong_predictions", ascending=False)
            .head(20)
        )

        plt.figure(figsize=(12, 6))
        sns.barplot(data=category_wrong, x="wrong_predictions", y="category")
        plt.title(f"Top Categories by Wrong Prediction Count\n{row['model']} | {row['dataset']}")
        plt.xlabel("Wrong predictions")
        plt.ylabel("Category")
        plt.tight_layout()
        plt.show()

        display(category_wrong)
